# ADS Bibliometric Analysis Pipeline

This notebook is the single entry point for the `ads_bib` package. It executes all steps sequentially, with configuration cells between steps so that downstream parameters can be set based on upstream results.

**Pipeline Steps:**
1. Search & Export — Query ADS API, resolve bibcodes to metadata
2. Translation — Detect languages, translate non-English titles/abstracts
3. Tokenization — Create `full_text`, tokenize with spaCy
4. AND — Optional external author name disambiguation
5. Topic Modeling & Curation — BERTopic + datamapplot + cluster removal
6. Citation Networks — Direct, Co-Citation, Bibliographic Coupling, Author Co-Citation

---
## Setup

In [ ]:
import time

from ads_bib.notebook import get_notebook_session

_pipeline_start = time.time()

In [ ]:
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)
formatter = logging.Formatter("%(message)s")

if root_logger.handlers:
    for handler in root_logger.handlers:
        handler.setLevel(logging.INFO)
        handler.setFormatter(formatter)
else:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)

logger = logging.getLogger("pipeline")

---
## Global Run Control

Set run-level switches here. Phase-specific parameters are configured in each phase section below.


In [ ]:
# -- RUN CONFIGURATION --
# Set RESET_SESSION=True when you want a fresh run directory.
RESET_SESSION = False
RUN = {
    "run_name": "ADS_Curation_Run",
    "random_seed": 42,
    "openrouter_cost_mode": "hybrid",
}

session = get_notebook_session(
    run_name=RUN["run_name"],
    reset=RESET_SESSION,
    start_time=_pipeline_start,
)
session.set_section("run", RUN)

---
# Phase 1: Search & Export

Query the NASA ADS API for publications matching a search string, then resolve all bibcodes (publications + references) into full metadata.

### 1.1 Search Configuration

Set query, years, and retrieval limits. These define the corpus scope for all downstream steps.

In [ ]:
# --- SEARCH CONFIGURATION ---
# Modify the query string for your research question.
# See: https://ui.adsabs.harvard.edu/help/search/search-syntax

Set_A = "docs(library/P0hyiLw0T8qsyHSBpWigJA)"
Set_B = f"citations({Set_A}) AND year:1915-2000"
Set_C = f"citations(citations({Set_A})) AND year:1915-2000"
Set_D = f"({Set_A}) OR ({Set_B}) OR ({Set_C})"
Set_T = "(docs(library/_-RCjJotRKCZC1PP03MqwA) AND year:1915-2000)"

gravity_relativity_terms = [
    '(general AND relativi*)', 'gravit*', '(allgemein* AND relativi*)',
    '"relativité générale"', '"teoria della relatività"',
    '"Gravité quantique"', '"Gravità quantistica"',
    '(einheitlich* AND feld*)', 'Quantengravitation', '"champ unifié"',
    '(unified AND field*)', '"quantum gravity"', 'cosmo*', 'Kosmo*',
]
Set_E = f"abs:({' OR '.join(gravity_relativity_terms)}) AND year:1915-2000"

SEARCH = {
    "query": f"({Set_D}) OR ({Set_T}) OR ({Set_E})",
    "ads_token": None,
    "refresh_search": True,
    "refresh_export": True,
}
# Example quick focus query:
SEARCH["query"] = 'author:"Hawking, S*"'
session.set_section("search", SEARCH)

Alternative search ideas:

- Quick focus query: `SEARCH["query"] = 'author:"Treder, H*"'`
- Broader historical build: combine `Set_A`, `Set_B`, `Set_C`, `Set_T`, and a wider anchored `Set_E`.
- Update only `SEARCH["query"]`, then rerun the search/export cells.


### 1.2 Execute Search

Run the ADS search and persist results to the run folder.

In [ ]:
session.run_stage("search")

### 1.3 Export & Resolve Metadata

Resolve publications and references to structured metadata tables.

In [ ]:
session.run_stage("export")

In [ ]:
if session.publications is not None:
    preview_cols = [
        c for c in ("Bibcode", "Year", "Author", "Title", "References")
        if c in session.publications.columns
    ]
    logger.info(
        "Phase 1 preview: publications=%s rows, columns=%s",
        f"{len(session.publications):,}",
        len(session.publications.columns),
    )
    if preview_cols:
        display(session.publications[preview_cols].head(10))
    else:
        display(session.publications.head(10))

---
# Phase 2: Translation

Detect non-English titles and abstracts with fasttext, then translate them using either OpenRouter (API) or a local HuggingFace model.

### 2.1 Translation Configuration

Choose provider, model, and translation target language.

In [ ]:
# -- TRANSLATION CONFIGURATION --
# Providers: openrouter, huggingface_api, llama_server, nllb
TRANSLATE = {
    "enabled": True,
    "provider": "openrouter",
    "model": "nvidia/nemotron-3-super-120b-a12b",
    "model_repo": None,
    "model_file": None,
    "model_path": None,
    "api_key": None,
    "max_workers": 10,
    "max_tokens": 2048,
    "fasttext_model": str(session.paths["models"] / "lid.176.bin"),
}
session.set_section("translate", TRANSLATE)


In [ ]:
# -- LOCAL LLAMA-SERVER CONFIGURATION --
LLAMA_SERVER = {
    "command": "llama-server",
    "host": "127.0.0.1",
    "port": None,
    "threads": None,
    "ctx_size": 4096,
    "gpu_layers": -1,
    "startup_timeout_s": 120.0,
    "reasoning": "off",
}
session.set_section("llama_server", LLAMA_SERVER)


### 2.2 Language Detection + Translation

Detect language tags and translate non-English titles and abstracts.

In [ ]:
session.run_stage("translate")

### 2.3 Preview Translated Fields

Inspect translated fields.

In [ ]:
if session.publications is not None:
    preview_cols = [
        c for c in ("Title", "Title_en", "Abstract", "Abstract_en")
        if c in session.publications.columns
    ]
    if preview_cols:
        display(session.publications[preview_cols].head(5))

---
# Phase 3: Tokenization

Create `full_text` (Title + Abstract) and tokenize publications with spaCy. References are skipped for runtime stability.

In [ ]:
import os

# -- TOKENIZATION CONFIGURATION --
TOKENIZE = {
    "enabled": True,
    "spacy_model": "en_core_web_md",
    "batch_size": 512,
    "n_process": min(max((os.cpu_count() or 1) - 1, 1), 8),
    "disable": ("ner", "parser", "textcat"),
    "fallback_model": "en_core_web_md",
    "auto_download": True,
}
session.set_section("tokenize", TOKENIZE)


In [ ]:
session.run_stage("tokenize")

In [ ]:
if session.refs is not None:
    logger.info(
        "References dataset available at %s",
        session.run.paths["data"] / "references.parquet",
    )

---
# Phase 4: Author Name Disambiguation

Optionally run an external AND package on the curated source datasets. If disabled, the pipeline saves a passthrough checkpoint and continues.

In [ ]:
AUTHOR_DISAMBIGUATION = {
    "enabled": False,
    "model_bundle": None,
    "dataset_id": None,
    "force_refresh": False,
    "infer_stage": "full",
}
session.set_section("author_disambiguation", AUTHOR_DISAMBIGUATION)

In [ ]:
session.run_stage("author_disambiguation")

---
# Phase 5: Topic Modeling & Curation

Use BERTopic or Toponymy to discover thematic clusters, visualize with datamapplot, then remove unwanted clusters to curate the dataset.

**Important:** Set parameters below based on your dataset size from Phase 1.

### 5.1 Embedding Configuration

Configure embedding provider/model and optional sampling. These settings strongly affect topic quality and runtime/cost.

| Provider | Model Examples | Cost | Notes |
|----------|----------------|------|-------|
| `local` | `google/embeddinggemma-300m` | None | Encoder used by the official `local_cpu` and `local_gpu` presets |
| `huggingface_api` | `Qwen/Qwen3-Embedding-8B` | Per-token | Encoder used by the official HF Inference API preset |
| `openrouter` | `qwen/qwen3-embedding-8b` | Per-token | Encoder used by the official OpenRouter preset |

**Caching**: Embeddings are cached to `data/cache/embeddings/` with SHA-256 fingerprint validation. Changing the dataset or model triggers automatic recomputation.


In [ ]:
# -- EMBEDDING CONFIGURATION --
# Providers: openrouter, huggingface_api, local
TOPIC_MODEL = {
    "sample_size": None,
    "embedding_provider": "openrouter",
    "embedding_model": "qwen/qwen3-embedding-8b",
    "embedding_api_key": None,
    "embedding_batch_size": 64,
    "embedding_max_workers": 20,
}
session.set_section("topic_model", TOPIC_MODEL)


### 5.2 Compute Embeddings

Create or load cached embeddings for `full_text`.

In [ ]:
logger.info(
    "Topic stages reuse shared snapshots plus the embedding and reduction caches."
)

In [ ]:
session.run_stage("embeddings")

### 5.3 Dimensionality Reduction Configuration

Two projections are computed: **5D** (HDBSCAN clustering input) and **2D** (visualization & KDE).

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| `pacmap` | Fast, good local/global balance | No `densmap` (density-preserving mode) |
| `umap` | Supports `densmap=True` for KDE, better hierarchical structure | Slower, sensitive to `n_neighbors` |

**Key parameters:**

| Parameter | Default | Guidance |
|-----------|---------|----------|
| `n_neighbors` | 80 (PaCMAP) / 80 (UMAP) | Higher = more global structure; lower = more local detail |
| `min_dist` | 0.05 (UMAP only) | Lower = tighter clusters in 2D. Library default 0.1 is too loose for bibliometric data |
| `metric` | `angular`/`cosine` | PaCMAP uses `angular` (auto-converted from `cosine`) |
| `densmap` | `False` (UMAP only) | Set `True` in `PARAMS_2D` if you plan KDE density analysis downstream |


> **Tip**: If you plan KDE analysis on the 2D coordinates, use `method="umap"` with
> `PARAMS_2D = dict(..., densmap=True)`. Without densmap, UMAP inflates dense regions
> in the 2D projection, distorting density estimates.


In [ ]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "reduction_method": "pacmap",
    "params_5d": {
        "n_neighbors": 80,
        "metric": "angular",
        "random_state": RUN["random_seed"],
    },
    "params_2d": {
        "n_neighbors": 80,
        "metric": "angular",
        "random_state": RUN["random_seed"],
    },
}
session.set_section("topic_model", TOPIC_MODEL)

In [ ]:
session.run_stage("reduction")

### 5.4 Clustering Configuration

HDBSCAN discovers topic clusters based on density in the 5D space.

| Parameter | Default | Guidance |
|-----------|---------|----------|
| `MIN_CLUSTER_SIZE` | `max(15, n_docs * 0.001)` | Controls granularity: ~0.1% of docs. Lower = more (smaller) topics |
| `min_samples` | 2–3 | Lower = fewer outliers but noisier clusters. Higher = stricter density |
| `cluster_selection_method` | `"eom"` | Excess of Mass: selects most persistent clusters |
| `cluster_selection_epsilon` | 0.02–0.05 | Absorbs border points; higher = larger clusters, fewer outliers |

**Backend choice:**
- `fast_hdbscan`: Fastest, but no `prediction_data` or `gen_min_span_tree` (no `approximate_predict()`).
- `hdbscan`: Supports `prediction_data=True` for predicting new documents and `gen_min_span_tree=True` for hierarchy analysis.

`BASE_MIN_CLUSTER_SIZE` is only used by Toponymy for micro-cluster granularity.


In [ ]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "clustering_method": "fast_hdbscan",
    "cluster_params": {},
    "toponymy_cluster_params": {},
    "toponymy_layer_index": "auto",
}
# Keep auto unless you need one fixed working-layer compatibility view; auto picks the coarsest available overview.
session.set_section("topic_model", TOPIC_MODEL)

### 5.5 Backend & LLM Configuration

**Backend matrix:**

| Backend | Clustering Input | LLM Providers | Notes |
|---------|-----------------|---------------|-------|
| `bertopic` | 5D reduced vectors | `local`, `llama_server`, `huggingface_api`, `openrouter` | Standard BERTopic + outlier reduction |
| `toponymy` | 5D reduced vectors | `local`, `llama_server`, `openrouter` | Hierarchical layers, richer labeling |

Toponymy is hierarchy-first: the canonical output is the full `topic_layer_*` hierarchy. `topic_id` and `Name` remain working-layer compatibility aliases, and `Topic_Layer_*` stays as a legacy map alias. Keep `toponymy_layer_index = "auto"` unless you intentionally want one fixed compatibility layer.

This repo no longer supports `toponymy_evoc`. A clean-room proof showed that the raw-embedding EVoC path depended on undeclared upstream runtime dependencies and a legacy standalone `evoc` pin, so the supported hierarchy backend here is `toponymy` only.

The shipped `hf_api.yaml` preset is BERTopic-oriented. Switch backend and provider settings explicitly before using Toponymy.

**Representation pipeline** (BERTopic):

| Model | Role | Configurable |
|-------|------|--------------|
| `POS` | Part-of-speech filtering (keep nouns, adjectives) | `pos_spacy_model` |
| `KeyBERT` | Semantic keyword re-ranking | Requires local embedding model |
| `MMR` | Maximal Marginal Relevance (diversity) | `mmr_diversity` (0–1) |
| LLM | Final topic label generation | Provider, model, prompt |

Default pipeline: **POS → KeyBERT → MMR → LLM** (sequential). Parallel models stored separately in `topic_aspects_` for comparison.

`MIN_DF` guidance: `max(1, min(5, n_docs // 100))`. Small samples need `min_df=1`; large corpora benefit from higher values to suppress noise terms.


### 5.5a LLM Prompt for Topic Labels

Choose a named prompt via `TOPIC_MODEL["llm_prompt_name"]` or provide a full override via `TOPIC_MODEL["llm_prompt"]`.

Available prompt names:

- `physics`: gravitational physics, astrophysics, cosmology
- `generic`: domain-agnostic scientific labeling


In [ ]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "llm_prompt_name": "physics",
    "llm_prompt": None,
}
session.set_section("topic_model", TOPIC_MODEL)

In [ ]:
TOPIC_MODEL = {
    **TOPIC_MODEL,
    "backend": "toponymy",  # choose: "bertopic" | "toponymy"
    # Small corpora often need lower Toponymy clustering thresholds.
    # Uncomment and tune if topic_fit fails with first-layer cluster errors.
    # "toponymy_cluster_params": {"min_clusters": 3, "base_min_cluster_size": 10},
    "llm_provider": "openrouter",
    "llm_model": "nvidia/nemotron-3-super-120b-a12b",
    "llm_model_repo": None,
    "llm_model_file": None,
    "llm_model_path": None,
    "llm_api_key": None,
    "bertopic_label_max_tokens": 128,
    "toponymy_local_label_max_tokens": 128,
    "pipeline_models": ["POS", "KeyBERT", "MMR"],
    "parallel_models": ["MMR", "POS", "KeyBERT"],
    "toponymy_embedding_model": None,
    "toponymy_max_workers": 10,
    "min_df": None,
    "outlier_threshold": 0.5,
}
session.set_section("topic_model", TOPIC_MODEL)


### 5.5b Fit Topic Model

Run the selected backend (BERTopic or Toponymy) on reduced vectors or raw embeddings, depending on the backend. This is the most compute/cost-intensive step of the pipeline.

In [ ]:
session.run_stage("topic_fit")

In [ ]:
if session.topic_info is not None:
    display(session.topic_info)

### 5.5c Build Topic DataFrame

Merge topic assignments, 2D coordinates, and embeddings into the main DataFrame.

In [ ]:
session.run_stage("topic_dataframe")

In [ ]:
if session.topic_df is not None:
    display(session.topic_df.head(10))

### 5.6 Visualize Topics

Render an interactive topic map. BERTopic uses one flat right-side `Topics` panel; Toponymy keeps the same panel but indents child topics hierarchically. You can still optionally enable the topic tree when you want the extra expert hierarchy panel.

In [ ]:
VISUALIZATION = {
    "enabled": True,
    "title": "ADS Bibliometric Map",
    "subtitle_template": "Topics labeled with {provider}/{model}",
    "dark_mode": True,
    "font_family": "Cinzel",
    "topic_tree": False,
}
session.set_section("visualization", VISUALIZATION)

In [ ]:
session.run_stage("visualize")

### 5.7 Curate Dataset

Review the cluster summary, then remove clusters that don't fit your research question. For Toponymy backends, prefer explicit `cluster_targets` so you can remove clusters from any hierarchy layer.

In [ ]:
CURATION = {
    "cluster_targets": [],
    "clusters_to_remove": [],
}
session.set_section("curation", CURATION)

In [ ]:
session.run_stage("curate")

### Curated Dataset Artifact

In [ ]:
logger.info(
    "Curated dataset artifact: %s",
    session.run.paths["data"] / "publications.parquet",
)

In [ ]:
if session.curated_df is not None:
    from ads_bib.curate import get_cluster_summary

    display(get_cluster_summary(session.curated_df))
    display(session.curated_df.head(10))

---
# Phase 6: Citation Networks

Compute citation networks from the curated dataset and export to GEXF (Gephi), Graphology JSON (Sigma.js), CSV, and/or Web of Science format.

### 6.1 Citation Configuration

Set network metrics, thresholds, filters, and export formats.

In [ ]:
CITATIONS = {
    "metrics": [
        "direct",
        "co_citation",
        "bibliographic_coupling",
        "author_co_citation",
    ],
    "min_counts": {
        "direct": 5,
        "co_citation": 20,
        "bibliographic_coupling": 20,
        "author_co_citation": 10,
    },
    "authors_filter": None,
    "output_format": "gexf",
}
session.set_section("citations", CITATIONS)

### 6.2 Build Node Table & Compute Networks

Build node metadata and compute selected citation networks.

In [ ]:
session.run_stage("citations")

### 6.3 Export Web of Science Format

Export the curated corpus in WOS text format for CiteSpace/VOSviewer.

In [ ]:
suffix = "_filtered" if session.config.citations.authors_filter else ""
logger.info(
    "WOS export artifact: %s",
    session.run.paths["data"] / f"download_wos_export{suffix}.txt",
)

---
# Summary

Final dataset statistics and output file listing.

In [ ]:
logger.info("Run artifacts: %s", session.run.paths["root"])

In [ ]:
session.save_summary()